<a href="https://colab.research.google.com/github/kienle141204/weather-forecast-by-graphcast/blob/main/weather_forecast_by_graphcast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install --upgrade https://github.com/deepmind/graphcast/archive/master.zip

     - 1.7 MB 4.4 MB/s 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.3/172.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.1/541.1 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 709.3/709.3 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.4/196.4 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 133.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 4.9 MB/s eta 0:00:00
  Created wheel for graphcast: filename=graphcast-0.2.0.dev0-py3-none-any.whl size=132343 sha256=79d46149246f5a7e84d7b6d379dd33e8

In [3]:
!pip uninstall -y shapely
!pip install shapely --no-binary shapely

Found existing installation: shapely 2.1.0
Uninstalling shapely-2.1.0:
  Successfully uninstalled shapely-2.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 8.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for shapely: filename=shapely-2.1.0-cp311-cp311-linux_x86_64.whl size=1187438 sha256=185c549d56fabc42bd59fafddf729554f0a6890456c6deae8a8ab4c014ec79fd
  Stored in directory: /root/.cache/pip/wheels/1c/af/c6/303cf0027549d1cd9c1fb6991d3bf37dd1c18c317cf1cef5b5
Successfully built shapely


In [4]:
!pip install cdsapi isodate netCDF4 pysolar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.3 MB/s eta 0:00:00


In [5]:
# @title Thêm các thư viện cần thiết

import cdsapi
import datetime
import functools
from google.cloud import storage
from graphcast import autoregressive, casting, checkpoint, data_utils as du, graphcast, normalization, rollout
import haiku as hk
import isodate
import jax
import math
import netCDF4
import numpy as np
import os
import pandas as pd
from pysolar.radiation import get_radiation_direct
from pysolar.solar import get_altitude
import pytz
from typing import Dict
import warnings
import xarray
import zipfile
warnings.filterwarnings('ignore')

In [6]:
# @title Điều kiện của dữ liệu đầu vào

from enum import Enum

class Constants:

    class CDSConstants(Enum):

        TIME_FIELD = 'valid_time'
        LAT_FIELD = 'latitude'
        LON_FIELD = 'longitude'
        PRESSURE_FIELD = 'pressure_level'

    class Graphcast(Enum):

        TIME_FIELD = 'time'
        LAT_FIELD = 'latitude'
        LON_FIELD = 'longitude'
        PRESSURE_FIELD = 'level'
        BATCH_FIELD = 'batch'

In [7]:
# @title Kết nối với Climate Data Stoge để lấy dữ liệu và kết nối model graphcast để tải trọng số
client = cdsapi.Client(
    url="https://cds.climate.copernicus.eu/api",
    key="e7e4835d-09de-461f-95b8-59af20bad476"
)

gcs_client = storage.Client.create_anonymous_client()

gcs_bucket = gcs_client.get_bucket('dm_graphcast')

2025-04-20 15:40:50,100 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
INFO:datapi.legacy_api_client:[2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-04-20 15:40:50,102 WARNING [2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.


In [35]:
# @title Định nghĩa và giải thích các thông số

singlelevelfields = {
                        'u10': '10m_u_component_of_wind',
                        'v10': '10m_v_component_of_wind',
                        't2m': '2m_temperature',
                        'z': 'geopotential',
                        'lsm': 'land_sea_mask',
                        'msl': 'mean_sea_level_pressure',
                        'tisr': 'toa_incident_solar_radiation',
                        'tp': 'total_precipitation'
                    }
pressurelevelfields = {
                        'u': 'u_component_of_wind',
                        'v': 'v_component_of_wind',
                        'z': 'geopotential',
                        'q': 'specific_humidity',
                        't': 'temperature',
                        'w': 'vertical_velocity'
                    }
predictionFields = [
                        'u_component_of_wind',
                        'v_component_of_wind',
                        'geopotential',
                        'specific_humidity',
                        'temperature',
                        'vertical_velocity',
                        '10m_u_component_of_wind',
                        '10m_v_component_of_wind',
                        '2m_temperature',
                        'mean_sea_level_pressure',
                        'total_precipitation_6hr'
                    ]
pressure_levels = [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000]
pi = math.pi
gap = 6
predictions_steps = 40
watts_to_joules = 3600
first_prediction = datetime.datetime(2025, 4, 12, 18, 0)

lat_range = np.arange(20, 22, 1)
lon_range = np.arange(105, 107, 1)

In [36]:
class AssignCoordinates:

    coordinates = {
                    '2m_temperature': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'lat', 'time'],
                    'mean_sea_level_pressure': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'lat', 'time'],
                    '10m_v_component_of_wind': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'lat', 'time'],
                    '10m_u_component_of_wind': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'lat', 'time'],
                    'total_precipitation_6hr': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'lat', 'time'],
                    'temperature': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'lat', 'level', 'time'],
                    'geopotential': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'lat', 'level', 'time'],
                    'u_component_of_wind': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'lat', 'level', 'time'],
                    'v_component_of_wind': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'lat', 'level', 'time'],
                    'vertical_velocity': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'lat', 'level', 'time'],
                    'specific_humidity': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'lat', 'level', 'time'],
                    'toa_incident_solar_radiation': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'lat', 'time'],
                    'year_progress_cos': [Constants.Graphcast.BATCH_FIELD.value, 'time'],
                    'year_progress_sin': [Constants.Graphcast.BATCH_FIELD.value, 'time'],
                    'day_progress_cos': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'time'],
                    'day_progress_sin': [Constants.Graphcast.BATCH_FIELD.value, 'lon', 'time'],
                    'geopotential_at_surface': ['lon', 'lat'],
                    'land_sea_mask': ['lon', 'lat'],
                }

In [37]:
# @title Tải Model

print('Connecting to dm_graphcast bucket...')
with gcs_bucket.blob(f'params/GraphCast_small - ERA5 1979-2015 - resolution 1.0 - pressure levels 13 - mesh 2to5 - precipitation input and output.npz').open('rb') as model:
    ckpt = checkpoint.load(model, graphcast.CheckPoint)
    params = ckpt.params
    state = {}
    model_config = ckpt.model_config
    task_config = ckpt.task_config

# Tải diffs_stddev_by_level.nc
print('Loading the diffs_stddev_by_level.nc file...')
blob = gcs_bucket.blob('stats/diffs_stddev_by_level.nc')
with blob.open('rb') as f:
    diffs_stddev_by_level = xarray.load_dataset(f).compute()

# Tải mean_by_level.nc
print('Loading the mean_by_level.nc file...')
blob = gcs_bucket.blob('stats/mean_by_level.nc')
with blob.open('rb') as f:
    mean_by_level = xarray.load_dataset(f).compute()

# Tải stddev_by_level.nc
print('Loading the stddev_by_level.nc file...')
blob = gcs_bucket.blob('stats/stddev_by_level.nc')
with blob.open('rb') as f:
    stddev_by_level = xarray.load_dataset(f).compute()

Connecting to dm_graphcast bucket...
Loading the diffs_stddev_by_level.nc file...
Loading the mean_by_level.nc file...
Loading the stddev_by_level.nc file...


In [38]:
# Khởi tạo mô hình GraphCast với các lớp bao bọc: Bfloat16, normalization và autoregressive
def construct_wrapped_graphcast(model_config: graphcast.ModelConfig, task_config: graphcast.TaskConfig):
    predictor = graphcast.GraphCast(model_config, task_config)
    predictor = casting.Bfloat16Cast(predictor)
    predictor = normalization.InputsAndResiduals(
        predictor,
        diffs_stddev_by_level=diffs_stddev_by_level,
        mean_by_level=mean_by_level,
        stddev_by_level=stddev_by_level
    )
    predictor = autoregressive.Predictor(predictor, gradient_checkpointing=True)
    return predictor

# Forward pass cho mô hình GraphCast, có lưu trạng thái nhờ Haiku
@hk.transform_with_state
def run_forward(model_config, task_config, inputs, targets_template, forcings):
    predictor = construct_wrapped_graphcast(model_config, task_config)
    return predictor(inputs, targets_template=targets_template, forcings=forcings)

# Truyền sẵn model_config và task_config vào hàm
def with_configs(fn):
    return functools.partial(fn, model_config=model_config, task_config=task_config)

# Truyền sẵn params và state vào hàm
def with_params(fn):
    return functools.partial(fn, params=params, state=state)

# Loại bỏ phần state trả về, chỉ giữ output chính
def drop_state(fn):
    return lambda **kw: fn(**kw)[0]

# Hàm forward đã được JIT, gắn sẵn config, params và bỏ phần state
run_forward_jitted = drop_state(
    with_params(
        jax.jit(
            with_configs(run_forward.apply)
        )
    )
)


In [39]:
# @title Định nghĩa lớp dự đoán

class Predictor:

    @classmethod
    def predict(cls, inputs, targets, forcings) -> xarray.Dataset:

        predictions = rollout.chunked_prediction(run_forward_jitted, rng = jax.random.PRNGKey(0), inputs = inputs, targets_template = targets, forcings = forcings)

        return predictions

In [40]:
# Chuyển đổi đối tượng ngày/thời gian hoặc chuỗi ISO thành datetime.datetime
def toDatetime(dt) -> datetime.datetime:
    if isinstance(dt, datetime.date) and isinstance(dt, datetime.datetime):
        return dt
    elif isinstance(dt, datetime.date) and not isinstance(dt, datetime.datetime):
        return datetime.datetime.combine(dt, datetime.datetime.min.time())
    elif isinstance(dt, str):
        if 'T' in dt:
            return isodate.parse_datetime(dt)
        else:
            return datetime.datetime.combine(isodate.parse_date(dt), datetime.datetime.min.time())

# Tạo mảng numpy có kích thước bất kỳ với toàn giá trị NaN
def nans(*args) -> list:
    return np.full((args), np.nan)

# Cộng khoảng thời gian (delta) vào datetime
def deltaTime(dt, **delta) -> datetime.datetime:
    return dt + datetime.timedelta(**delta)

# Thêm hoặc chuyển đổi timezone cho đối tượng datetime
def addTimezone(dt, tz=pytz.UTC) -> datetime.datetime:
    dt = toDatetime(dt)
    if dt.tzinfo == None:
        return pytz.UTC.localize(dt).astimezone(tz)
    else:
        return dt.astimezone(tz)

# Loại bỏ các cột không cần thiết khỏi DataFrame (ví dụ: 'number', 'expver')
def remove_junk_columns(df: pd.DataFrame):
    for col in ['number', 'expver']:
        if col in df.columns.values.tolist():
            df.pop(col)
    return df

# Đọc và xử lý các file dữ liệu single-level từ file zip NetCDF
def getSingleLevelValues(filename):
    extract_to = filename.split('.')[0]
    with zipfile.ZipFile(filename, 'r') as f:
        f.extractall(extract_to)

    dfs = []
    for i in os.listdir(extract_to):
        extension = i.split('.')[-1]
        if extension == 'nc':
            df = xarray.open_dataset('{}/{}'.format(extract_to, i), engine=netCDF4.__name__.lower()).to_dataframe()
            df = remove_junk_columns(df)
            dfs.append(df)

    single_level_df = pd.concat(dfs, axis=1)
    return single_level_df


In [41]:
# @title Định nghĩa hàm tải dữ liệu từ CDS

# Tải và xử lý dữ liệu ERA5 bao gồm dữ liệu bề mặt (single level) và áp suất (pressure level)
def getSingleAndPressureValues():
    client.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': list(singlelevelfields.values()),
            'grid': '1.0/1.0',
            'year': [2025],
            'month': [4],
            'day': [12],
            'time': ['06:00', '12:00'],
            "area": [22, 105, 20, 107],
            'data_format': 'netcdf',
            'download_format': 'zip'
        }
    ).download('single-level.zip')

    singlelevel = getSingleLevelValues('single-level.zip')
    singlelevel = singlelevel.rename(columns={col: singlelevelfields[col] for col in singlelevel.columns.values.tolist() if col in singlelevelfields})
    singlelevel = singlelevel.rename(columns={'geopotential': 'geopotential_at_surface'})

    # Tính tổng lượng mưa 6 giờ gần nhất
    singlelevel = singlelevel.sort_index()
    singlelevel['total_precipitation_6hr'] = singlelevel.groupby(level=[0, 1])['total_precipitation'].rolling(window=6, min_periods=1).sum().reset_index(level=[0, 1], drop=True)
    singlelevel.pop('total_precipitation')

    client.retrieve(
        'reanalysis-era5-pressure-levels',
        {
            'product_type': 'reanalysis',
            'variable': list(pressurelevelfields.values()),
            'grid': '1.0/1.0',
            'year': [2025],
            'month': [4],
            'day': [12],
            'time': ['06:00', '12:00'],
            "area": [22, 105, 20, 107],
            'pressure_level': pressure_levels,
            'data_format': 'netcdf',
            'download_format': 'unarchived'
        }
    ).download('pressure-level.nc')

    pressurelevel = xarray.open_dataset('pressure-level.nc', engine=netCDF4.__name__.lower()).to_dataframe()
    pressurelevel = remove_junk_columns(pressurelevel)
    pressurelevel = pressurelevel.rename(columns={col: pressurelevelfields[col] for col in pressurelevel.columns.values.tolist() if col in pressurelevelfields})

    return singlelevel, pressurelevel


In [42]:
# Thêm thông tin tiến độ của năm (dạng sin/cos) vào DataFrame
def addYearProgress(secs, data):
    progress = du.get_year_progress(secs)
    data['year_progress_sin'] = math.sin(2 * pi * progress)
    data['year_progress_cos'] = math.cos(2 * pi * progress)
    return data

# Thêm thông tin tiến độ của ngày (dạng sin/cos) theo kinh độ vào DataFrame
def addDayProgress(secs, lon: str, data: pd.DataFrame):
    lons = data.index.get_level_values(lon).unique()
    progress: np.ndarray = du.get_day_progress(secs, np.array(lons))
    prxlon = {lon: prog for lon, prog in list(zip(list(lons), progress.tolist()))}
    data['day_progress_sin'] = data.index.get_level_values(lon).map(lambda x: math.sin(2 * pi * prxlon[x]))
    data['day_progress_cos'] = data.index.get_level_values(lon).map(lambda x: math.cos(2 * pi * prxlon[x]))
    return data

# Tích hợp thông tin tiến độ ngày và năm vào toàn bộ dữ liệu
def integrateProgress(data: pd.DataFrame):
    for dt in data.index.get_level_values(Constants.CDSConstants.TIME_FIELD.value).unique():
        seconds_since_epoch = toDatetime(dt).timestamp()
        data = addYearProgress(seconds_since_epoch, data)
        data = addDayProgress(seconds_since_epoch, 'longitude' if 'longitude' in data.index.names else 'lon', data)
    return data

# Tính bức xạ mặt trời chiếu tới bề mặt dựa trên vị trí và thời gian
def getSolarRadiation(longitude, latitude, dt):
    altitude_degrees = get_altitude(latitude, longitude, addTimezone(dt))
    solar_radiation = get_radiation_direct(dt, altitude_degrees) if altitude_degrees > 0 else 0
    return solar_radiation * watts_to_joules

# Tích hợp giá trị bức xạ mặt trời vào DataFrame
def integrateSolarRadiation(data: pd.DataFrame):
    dates = list(data.index.get_level_values(Constants.CDSConstants.TIME_FIELD.value).unique())
    coords = [[lat, lon] for lat in lat_range for lon in lon_range]
    values = []

    for dt in dates:
        values.extend(list(map(lambda coord: {
            Constants.CDSConstants.TIME_FIELD.value: dt,
            Constants.CDSConstants.LON_FIELD.value: coord[1],
            Constants.CDSConstants.LAT_FIELD.value: coord[0],
            'toa_incident_solar_radiation': getSolarRadiation(coord[1], coord[0], dt)
        }, coords)))

    values = pd.DataFrame(values).set_index(
        keys=[Constants.CDSConstants.LAT_FIELD.value, Constants.CDSConstants.LON_FIELD.value, Constants.CDSConstants.TIME_FIELD.value]
    )

    return pd.merge(data, values, left_index=True, right_index=True, how='inner')

# Điều chỉnh các toạ độ không cần thiết trong xarray.Dataset
def modifyCoordinates(data: xarray.Dataset):
    for var in list(data.data_vars):
        varArray: xarray.DataArray = data[var]
        nonIndices = list(set(list(varArray.coords)).difference(set(AssignCoordinates.coordinates[var])))
        data[var] = varArray.isel(**{coord: 0 for coord in nonIndices})
    data = data.drop_vars(Constants.Graphcast.BATCH_FIELD.value)
    return data

# Chuyển đổi DataFrame thành xarray.Dataset với tên trục được chuẩn hoá
def makeXarray(data: pd.DataFrame) -> xarray.Dataset:
    data = data.rename_axis(index={
        Constants.CDSConstants.TIME_FIELD.value: Constants.Graphcast.TIME_FIELD.value,
        Constants.CDSConstants.PRESSURE_FIELD.value: Constants.Graphcast.PRESSURE_FIELD.value
    })
    data = data.to_xarray()
    data = modifyCoordinates(data)
    return data

# Chuẩn hoá dữ liệu đầu vào về đúng định dạng chỉ mục, thêm batch index nếu chưa có
def formatData(data: pd.DataFrame) -> pd.DataFrame:
    data = data.rename_axis(index={Constants.CDSConstants.LAT_FIELD.value: 'lat', Constants.CDSConstants.LON_FIELD.value: 'lon'})
    if Constants.Graphcast.BATCH_FIELD.value not in data.index.names:
        data[Constants.Graphcast.BATCH_FIELD.value] = 0
        data = data.set_index(Constants.Graphcast.BATCH_FIELD.value, append=True)
    return data

# Tạo khung dữ liệu rỗng chứa các trường dự đoán mục tiêu (target)
def getTargets(dt, data: pd.DataFrame):
    lat = sorted(data.index.get_level_values('lat').unique().tolist())
    lon = sorted(data.index.get_level_values('lon').unique().tolist())
    levels = sorted(data.index.get_level_values('pressure_level').unique().tolist())
    batch = data.index.get_level_values(Constants.Graphcast.BATCH_FIELD.value).unique().tolist()
    time = [deltaTime(dt, hours=days * gap) for days in range(predictions_steps)]

    target = xarray.Dataset({
        field: (['lat', 'lon', 'level', Constants.CDSConstants.TIME_FIELD.value], nans(len(lat), len(lon), len(levels), len(time)))
        for field in predictionFields
    }, coords={
        'lat': lat,
        'lon': lon,
        'level': levels,
        Constants.CDSConstants.TIME_FIELD.value: time,
        Constants.Graphcast.BATCH_FIELD.value: batch
    })

    return target.to_dataframe()

# Tạo forcing dataframe từ dữ liệu gốc bằng cách loại các trường dự đoán và thêm thông tin thời gian, bức xạ
def getForcings(data: pd.DataFrame):
    forcingdf = data.reset_index(level='level', drop=True).drop(labels=predictionFields, axis=1)
    forcingdf = pd.DataFrame(index=forcingdf.index.drop_duplicates(keep='first'))
    forcingdf = integrateProgress(forcingdf)
    forcingdf = integrateSolarRadiation(forcingdf)
    return forcingdf


In [43]:
# @title Tải dữ liệu và gộp nó vào thành định dạng thích hợp cho mô hình graphcast

values:Dict[str, xarray.Dataset] = {}

single, pressure = getSingleAndPressureValues()
values['inputs'] = pd.merge(pressure, single, left_index = True, right_index = True, how = 'inner')
values['inputs'] = integrateProgress(values['inputs'])
values['inputs'] = formatData(values['inputs'])
values['targets'] = getTargets(first_prediction, values['inputs'])
values['forcings'] = getForcings(values['targets'])
values = {value:makeXarray(values[value]) for value in values}

2025-04-20 15:48:08,679 INFO Request ID is 7d2bb555-b5f5-41ee-bc1a-f1f63252b9dc
INFO:datapi.legacy_api_client:Request ID is 7d2bb555-b5f5-41ee-bc1a-f1f63252b9dc
2025-04-20 15:48:08,884 INFO status has been updated to accepted
INFO:datapi.legacy_api_client:status has been updated to accepted
2025-04-20 15:48:23,239 INFO status has been updated to successful
INFO:datapi.legacy_api_client:status has been updated to successful


86f097c75a19b99924b514f0a4fa8e74.zip:   0%|          | 0.00/104k [00:00<?, ?B/s]

2025-04-20 15:48:26,160 INFO Request ID is 58e7ac9f-685f-4246-bde6-92bf116d37a7
INFO:datapi.legacy_api_client:Request ID is 58e7ac9f-685f-4246-bde6-92bf116d37a7
2025-04-20 15:48:26,354 INFO status has been updated to accepted
INFO:datapi.legacy_api_client:status has been updated to accepted
2025-04-20 15:48:35,222 INFO status has been updated to successful
INFO:datapi.legacy_api_client:status has been updated to successful


394591d72638072330a91a9c4eb7fb83.nc:   0%|          | 0.00/78.2k [00:00<?, ?B/s]

In [44]:
import jax
print(jax.devices())

[CudaDevice(id=0)]


In [45]:
# @title Tiến hành dự đoán

predictions = Predictor.predict(values['inputs'], values['targets'], values['forcings'])
predictions.to_dataframe().to_csv('predictions_hanoi_10d.csv', sep = ',')

In [46]:
predictions

<xarray.Dataset> Size: 120kB
Dimensions:                  (time: 40, lat: 3, lon: 3, batch: 1, level: 13)
Coordinates:
  * lat                      (lat) float64 24B 20.0 21.0 22.0
  * lon                      (lon) float64 24B 105.0 106.0 107.0
  * level                    (level) float64 104B 50.0 100.0 ... 925.0 1e+03
  * time                     (time) datetime64[ns] 320B 2025-04-12T18:00:00 ....
Dimensions without coordinates: batch
Data variables:
    10m_u_component_of_wind  (time, lat, lon, batch) float32 1kB -2.485 ... -...
    10m_v_component_of_wind  (time, lat, lon, batch) float32 1kB -1.095 ... 0...
    2m_temperature           (time, lat, lon, batch) float32 1kB 291.1 ... 289.2
    geopotential             (time, lat, lon, level, batch) float32 19kB 2.01...
    mean_sea_level_pressure  (time, lat, lon, batch) float32 1kB 1.015e+05 .....
    specific_humidity        (time, lat, lon, level, batch) float32 19kB 2.95...
    temperature              (time, lat, lon, level, batch) float32 19kB 207....
    total_precipitation_6hr  (time, lat, lon, batch) float32 1kB 0.07874 ... ...
    u_component_of_wind      (time, lat, lon, level, batch) float32 19kB -3.7...
    v_component_of_wind      (time, lat, lon, level, batch) float32 19kB -0.2...
    vertical_velocity        (time, lat, lon, level, batch) float32 19kB -0.0...

In [48]:
# @title Lưu kết quả

# Đặt đường dẫn đến file credentials
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = '/content/silicon-stock-452315-h4-1159b7c155af.json'
# Khởi tạo client
c = storage.Client()

bucket_name = 'weather_test_123'
destination_blob_name = 'predictions_hanoi_10d.csv'  # Tên file sẽ lưu trên GCS

bucket = c.get_bucket(bucket_name)

blob = bucket.blob(destination_blob_name)
blob.upload_from_filename('predictions_hanoi_10d.csv')
print(f"Lưu thành công file {destination_blob_name} vào bucket {bucket_name}")

Lưu thành công file predictions_hanoi_10d.csv vào bucket weather_test_123
